# Hermes Agent: 4계층 메모리 아키텍처 및 Closed Learning Loop 실습

본 노트북은 **Hermes Agent**의 혁신적인 **4계층 메모리 구조(4-Layer Memory Architecture)**와 **Closed Learning Loop(자가 학습 순환 구조)**를 `frontier-agent-lab` 환경에서 직접 실습하고 검증하기 위해 제작되었습니다.

---

### 🎯 학습 목표
1. **L1 ~ L4 메모리 계층 구조 이해**: Short-term Working Memory부터 Long-term Episodic/Semantic Memory 및 Procedural Memory까지의 분리와 역할을 이해합니다.
2. **L3 Semantic Memory 스토어 제어**: `§` 구분자 기반의 `MEMORY.md` / `USER.md` 스토리지, **Frozen Snapshot** 패턴, 용량 제한(Capacity Bounding) 동작을 확인합니다.
3. **L2 Episodic Memory & Lineage 인출**: SQLite FTS5 기반의 세션 요약 검색과 메시지 **Anchor 선정 + Bookend (첫 3개 + tail 3개) Lineage 인출** 메커니즘을 경험합니다.
4. **Closed Learning Loop & Background Review**: `after_agent` 미들웨어 훅에서 비동기 데몬 스레드로 LLM이 대화를 리뷰하여 장기 메모리를 자동 갱신하는 과정을 검증합니다.

## 🛠️ Step 0. 환경 세팅 (Environment Setup)

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`), 그리고 통합 LLM 팩토리를 초기화합니다.

In [1]:
import os
import sys
import re
import json
import time
import asyncio
import warnings
import nest_asyncio
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# 1. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 2. 프로젝트 루트 상향 동적 탐색 (어느 서브 폴더에 노트북이 있어도 100% 작동)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. 환경변수 명시적 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path, override=True)

# LangSmith / LangChain Tracing 경고 제어
os.environ["LANGCHAIN_TRACING_V2"] = "false"

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 4. 통합 Chat Model Factory 및 핵심 모듈 임포트
from app.utils import init_chat_model, normalize_content
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from modules.common.agent_tracer import AgentTracer

# 5. 이전 코드 호환용 get_llm 헬퍼 (노트북 내 잔여 get_llm 호출 완충)
def get_llm(model_name="openai:gpt-4o-mini", **kwargs):
    return init_chat_model(model=model_name, **kwargs)

# 6. 기본 LLM 및 트레이서 초기화
llm = init_chat_model(model="openai:gpt-4o-mini", temperature=0.0)
tracer = AgentTracer(log_dir="./artifacts/logs", verbose=True)

print(f"✅ LLM Model Initialized: {getattr(llm, 'model', getattr(llm, 'model_name', 'openai:gpt-4o-mini'))}")
print("✅ Setup completed successfully!")

✅ Working Directory: /mnt/c/Users/hyoun/Desktop/working_project/frontier_agent_analysis
✅ Project Root: /mnt/c/Users/hyoun/Desktop/working_project/frontier_agent_analysis
✅ LLM Model Initialized: gpt-4o-mini
✅ Setup completed successfully!


## 🏗️ 4계층 메모리 & Closed Learning Loop 전체 구조

Hermes 에이전트는 대화 흐름을 방해하지 않으면서 턴(Turn) 진행 중 및 세션 완료 후 메모리를 추출하여 장기 기억으로 전환합니다.
특히 **순정 Hermes 아키텍처(A안: Pull-only)**는 프롬프트 오염을 막기 위한 **Clean Slate 원칙**과 **KV-Cache 최적화**를 핵심으로 삼습니다.

![Hermes Agent 4-Layer Memory Architecture](assets/hermes_memory_architecture.svg)

### 📌 순정 Hermes 4계층 메모리 인출 모델 (A안: Pull-only)
| 계층 | 명칭 | 저장소 / 기술 | 인출 방식 (Retrieval) | 설계 철학 |
|:---|:---|:---|:---|:---|
| **L1** | **Working Memory** | `AsyncSqliteSaver` (체크포인터) | LangGraph 상태 자동 복원 | 현재 활성 세션의 단기 대화 상태 |
| **L2** | **Episodic Memory** | `EpisodicStore` (SQLite FTS5) | **도구 2단계 인출 (Pull)**: `session_search` ➔ `session_recall` | **Clean Slate 원칙**: 프롬프트 자동 주입(Push)을 하지 않아 컨텍스트 오염 방지 |
| **L3** | **Semantic Memory** | `SemanticMemoryStore` (`MEMORY.md`, `USER.md`) | **시스템 프롬프트 상시 주입 (Push)**: 세션 시작 시 **Frozen Snapshot** 주입 | 세션 도중 프롬프트가 변경되지 않아 **GPU KV-Cache HIT 극대화** |
| **L4** | **Procedural Memory**| `PromptAssembler` | 정적 프롬프트 조립 & Caching Boundary | 불변 헌법, 도구 명세, 가이드라인 |

> 💡 **Closed Learning Loop**: 대화 턴이 끝난 후 `after_agent` 훅이 비동기 데몬 스레드(`_spawn_background_review`)를 구동하여, 별도 LLM이 대화를 검토하고 L3 마크다운(`memory` add/replace)과 L2 세션 요약(`finalize_session`)을 사용자 응답 지연(Latency) 없이 백그라운드에서 저장합니다.


## 1. L3 Semantic Memory (의미론적 장기 기억스토어)

**L3 Semantic Memory**는 에이전트가 지속적으로 보유해야 하는 **세상에 대한 팩트(`MEMORY.md`)**와 **사용자에 대한 프로필/선호도(`USER.md`)**를 보관하는 장기 기억 장치입니다.

--- 

### 🛠️ Semantic Memory CRUD Actions (`memory` 도구 동작 메커니즘)

L3 메모리 스토어는 용량 무한 증식을 방지하고 기억의 질을 높이기 위해 **3가지 액션(Action)**을 지원하며, LLM이 백그라운드 리뷰 시 이를 **자율적으로 판단하여 격발**합니다.

| Action | 설명 및 사용 시점 | 저장/동작 방식 |
|---|---|---|
| **`add`** | 기존 기억에 없는 **완전히 새로운 독립적 팩트/선호도**가 등장했을 때 사용 | `§` (Section Sign) 구분자로 독립된 엔트리 추가 |
| **`replace`** | 기존 팩트와 연관되거나 업그레이드된 정보가 등장했을 때 **스마트 병합(Merge & Compact)** | 기존 `old_text`를 찾아 문장을 한 덩어리로 압축·수정 (`§` 개수 유지) |
| **`remove`** | 기존 팩트가 폐기되거나 더 이상 유효하지 않게 되었을 때 **기억 삭제** | 대상 엔트리를 찾아 스토어에서 완전 제거 |

--- 

### 1.1 L3 Semantic Memory의 생성 (Generation & Storage)

L1 Working Memory(대화 맥락)에서 L3 Semantic Memory로 팩트가 생성되어 기록되는 통로는 **2가지**가 있습니다.

1. **에이전트 자율 생성 (`memory` 도구 직접 호출)**:
   - 대화 진행 중 에이전트가 중요한 정보를 전달받으면 자율적으로 `@tool memory(action='add', target='memory', content='...')` 도구를 격발하여 저장합니다.

2. **Closed Learning Loop 자동 생성 (`MemoryMiddleware` 백그라운드 학습)**:
   - 턴이 끝난 후 `after_agent` 미들웨어 훅에서 비동기 데몬 스레드가 작동합니다.
   - LLM이 최근 L1 대화 내역을 관찰하여 신규 팩트나 유저 선호도를 자동으로 추출(Fact Extraction)하고 XML 태그(`<call:memory ... />`) 또는 함수 형태로 파싱하여 파일에 저장합니다.

아래 실습 코드에서 두 가지 생성 방식을 모두 시연합니다.

In [2]:
import tempfile
import os
from app.middleware.memory import SemanticMemoryStore, EpisodicStore, MemoryMiddleware
from app.utils import get_llm

print("=== [1.1 L3 Semantic Memory 생성 실습] ===\n")

# 1. 실습용 임시 디렉토리 생성 및 스토어 초기화
tmp_dir = tempfile.mkdtemp()
semantic_store = SemanticMemoryStore(memory_dir=tmp_dir)
semantic_store.load_from_disk() # MEMORY.md와 USER.md를 읽습니다. 현재는 비어있습니다.

# -----------------------------------------------------
# [방법 1] 에이전트 도구를 통한 직접 생성 (Direct Tool Call)
# -----------------------------------------------------
print("📌 [방법 1] 스토어 add() 메서드를 통한 팩트 직접 저장")
res1 = semantic_store.add("memory", "Project runs on Python 3.12 + LangChain 0.3.")
res2 = semantic_store.add("user", "User prefers Korean responses with technical terms in English.")
print(f"  - MEMORY.md 저장 결과: {res1['message']} (사용량: {res1['usage']})")
print(f"  - USER.md 저장 결과:   {res2['message']} (사용량: {res2['usage']})\n")

print("\n📂 [1차로 생성된 MEMORY.md 파일 내용]")
with open(os.path.join(tmp_dir, "MEMORY.md"), "r", encoding="utf-8") as f:
    print(f.read())

print("📂 [최종 생성된 USER.md 파일 내용]")
with open(os.path.join(tmp_dir, "USER.md"), "r", encoding="utf-8") as f:
    print(f.read())


=== [1.1 L3 Semantic Memory 생성 실습] ===

📌 [방법 1] 스토어 add() 메서드를 통한 팩트 직접 저장
  - MEMORY.md 저장 결과: Entry added. (사용량: 2% — 44/2,200 chars)
  - USER.md 저장 결과:   Entry added. (사용량: 4% — 62/1,375 chars)


📂 [1차로 생성된 MEMORY.md 파일 내용]
Project runs on Python 3.12 + LangChain 0.3.
📂 [최종 생성된 USER.md 파일 내용]
User prefers Korean responses with technical terms in English.


In [3]:
# -----------------------------------------------------
# [방법 2] Closed Learning Loop를 통한 L1 대화 자동 팩트 추출
# -----------------------------------------------------
print("📌 [방법 2] L1 대화 내역으로부터 LLM 백그라운드 자동 팩트 추출")
episodic_store = EpisodicStore(os.path.join(tmp_dir, "episodic.db"))
middleware = MemoryMiddleware(semantic_store=semantic_store, episodic_store=episodic_store, review_llm=llm)

# 유저가 새로운 정보(테스트 프레임워크 선호도)를 밝힌 L1 대화 샘플
l1_conversation = [
    {"role": "user", "content": "앞으로 파이썬 코드를 작성할 때는 단위 테스트로 zawsze pytest를 써줘."},
    {"role": "assistant", "content": "네, 알겠습니다! pytest를 기본 테스트 프레임워크로 기억하겠습니다."}
]

# 백그라운드 리뷰 실행 (L1 대화 리뷰 -> LLM이 팩트 추출 -> memory() 자동 실행)
middleware._review_semantic_memory(l1_conversation)

# -----------------------------------------------------
# [결과 확인] 생성된 디스크 파일(MEMORY.md / USER.md)의 내용 조회
# -----------------------------------------------------
print("\n📂 [최종 생성된 MEMORY.md 파일 내용]")
with open(os.path.join(tmp_dir, "MEMORY.md"), "r", encoding="utf-8") as f:
    print(f.read())

print("📂 [최종 생성된 USER.md 파일 내용]")
with open(os.path.join(tmp_dir, "USER.md"), "r", encoding="utf-8") as f:
    print(f.read())

📌 [방법 2] L1 대화 내역으로부터 LLM 백그라운드 자동 팩트 추출

📂 [최종 생성된 MEMORY.md 파일 내용]
Project runs on Python 3.12 + LangChain 0.3.
📂 [최종 생성된 USER.md 파일 내용]
User prefers Korean responses with technical terms in English.
§
User prefers to use pytest for unit testing in Python code.


### 🤖 LLM은 어떻게 `add` 대신 `replace` 병합을 자율 판단할까?

`MemoryMiddleware`는 백그라운드 리뷰 시 LLM에게 현재 `MEMORY.md` 및 `USER.md`에 저장된 최신 팩트 목록을 프롬프트로 보여줍니다.
LLM은 지침(`Merge overlapping entries using memory(action="replace", ...)`)에 따라 다음과 같이 자율 판단합니다:

- **상황 예시**: `MEMORY.md`에 이미 `Project runs on Python 3.12 + LangChain 0.3.`이 있는 상태에서 유저가 *"pytest를 써줘"*라고 대화함.
- **LLM의 스마트 판단**: 단순 `add`를 남발하면 메모리가 지저분하게 파편화되므로, LLM이 `replace` 액션을 선택하여 `old_text="Project runs on Python 3.12"`를 `"Project runs on Python 3.12 + LangChain 0.3; user prefers pytest for unit testing."` 처럼 **세미콜론(`;`)으로 한 엔트리에 예쁘게 병합 요약**합니다!
- **결과적인 이점**: 독립된 팩트는 `§` 구분자로 늘어나고, 연관된 팩트는 세미콜론 `;`으로 압축 병합되어 **제한된 용량(`memory_char_limit: 2200`)을 효율적으로 활용**하게 됩니다.

### 1.2 L3 Semantic Memory의 인출 (Retrieval & Prompt/Tool Injection)

생성된 L3 메모리는 **2가지 방식**으로 인출되어 사용됩니다.

1. **시스템 프롬프트 자동 인출 & Frozen Snapshot 패턴**:
   - 세션이 시작될 때 `load_from_disk()`가 디스크의 마크다운 팩트를 읽어 **Frozen Snapshot(프롬프트 주입용 스냅샷)** 을 만듭니다.
   - 대화 진행 중 메모리가 새로 추가되어도 당해 세션 동안은 시스템 프롬프트가 변경되지 않고 **Frozen Snapshot**이 유지되어 **LLM Prefix Cache(프리픽스 캐시)** 비용 절감 효과를 제공합니다.
   - 새 팩트는 **다음 세션 시작 시 재로드되어 시스템 프롬프트에 자동으로 주입**됩니다.

2. **도구(Tool)를 통한 동적 조회 및 관리**:
   - 필요 시 에이전트는 `memory(action='replace', ...)` 또는 `memory(action='remove', ...)` 도구를 사용하여 기존 기억을 교체하거나 삭제할 수 있습니다.

아래 실습 코드에서 세션 간 Frozen Snapshot 인출 전이와 도구 기반 기억 수정을 확인해보세요.

[1] 시작 시점의 시스템 프롬프트 주입 스냅샷 확인

위 두 번의 실행으로 MEMORY.md와 USER.md의 내용이 생성됐지만, 이번 세션에서는 변경 사항이 프롬프트 캐싱을 위해 시스템 프롬프트에 영향을 주지 않습니다.
어차피 해당 정보는 대화 컨텍스트에 이미 있기 때문입니다.

In [4]:
print("📌 [세션 1 시작 시점] 시스템 프롬프트에 주입될 Frozen Snapshot:")
prompt_snapshot_session1 = semantic_store.format_for_prompt("memory")
print(prompt_snapshot_session1)
print("\n💡 설명: 세션 1이 시작할 당시의 스냅샷이 유지되므로, 세션 도중 새로 추가된 팩트는 세션 1의 시스템 프롬프트에 영향을 주지 않습니다. (Prefix Cache 보존!)\n")

📌 [세션 1 시작 시점] 시스템 프롬프트에 주입될 Frozen Snapshot:
None

💡 설명: 세션 1이 시작할 당시의 스냅샷이 유지되므로, 세션 도중 새로 추가된 팩트는 세션 1의 시스템 프롬프트에 영향을 주지 않습니다. (Prefix Cache 보존!)



### [2] 새로운 세션을 시작할 때 로드 -> 프롬프트 주입 스냅샷 전이 확인

새로운 세션이 시작됐다고 가정합니다. 이때 아래와 같이 문서를 읽게 되면, 에이전트가 기억을 떠올리는 겁니다.

In [5]:
print("📌 [세션 2 시작 시점] 새로운 세션 시작 후 load_from_disk() 실행 결과:")
session2_store = SemanticMemoryStore(memory_dir=tmp_dir)
session2_store.load_from_disk() # 여기서 MEMORY.md와 USER.md를 읽습니다.

📌 [세션 2 시작 시점] 새로운 세션 시작 후 load_from_disk() 실행 결과:


이전 세션에서 저장되었던 모든 팩트가 세션 2의 시스템 프롬프트 Layer 4에 정상적으로 자동 인출되어 주입되었습니다!

In [6]:
prompt_snapshot_session2 = session2_store.format_for_prompt("memory")
print(prompt_snapshot_session2)

══════════════════════════════════════════════
MEMORY (agent's personal notes) [2% — 44/2,200 chars]
══════════════════════════════════════════════
Project runs on Python 3.12 + LangChain 0.3.


### [3] 도구를 통한 인출 및 기억 수정/삭제 (Replace / Remove)

에이전트는 도구를 통해 기억을 변경할 수도 있습니다. 에이전트가 시간이 지나 바뀐 기술 정보(버전 업그레이드 등)를 능동적으로 수정(replace)하는 과정을 보여줍니다.

In [7]:
print("📌 [도구 활용] 기존 팩트의 수정(Replace) 및 삭제(Remove)")
# 팩트 수정: Python 3.12 -> Python 3.12.3 with LangChain 0.3.5
replace_res = session2_store.replace(
    target="memory",
    old_text="Python 3.12",
    new_content="Project runs on Python 3.12.3 with LangChain 0.3.5; user prefers pytest for unit testing."
)
print(f"  - Replace 수정 결과: {replace_res['message']}")

📌 [도구 활용] 기존 팩트의 수정(Replace) 및 삭제(Remove)
  - Replace 수정 결과: Entry replaced.


In [8]:
print("\n📂 [수정 후 최신 MEMORY.md 파일 내용]")
with open(os.path.join(tmp_dir, "MEMORY.md"), "r", encoding="utf-8") as f:
    print(f.read())


📂 [수정 후 최신 MEMORY.md 파일 내용]
Project runs on Python 3.12.3 with LangChain 0.3.5; user prefers pytest for unit testing.


## 2. L2 Episodic Memory (세션 DB & Lineage 기반 Anchor 인출)

**L2 Episodic Memory**는 과거 세션 대화 전체를 SQLite(`episodic.db`)에 영구 보관하고 에이전트가 필요 시 도구를 통해 인출하는 에피소드 기억 시스템입니다.

### 🔑 핵심 메커니즘 (순정 Hermes 2단계 인출: Two-Stage Retrieval)
순정 Hermes는 프롬프트에 과거 세션을 매 턴 무차별 자동 주입하지 않고, 에이전트가 필요할 때 **2단계 도구 연계**로 인출합니다:

1. **1단계: 세션 탐색 (`session_search` 도구)**
   - **Discovery 모드**: 쿼리를 넘기면 세션 요약(English Summary)과 키워드(Keywords)를 기반으로 FTS5 전문 검색을 수행하여 관련 세션 후보(`session_id`, `summary`, `keywords`)를 반환합니다.
   - **Browse 모드**: 쿼리 없이 호출하면 가장 최근의 세션 목록을 최신순으로 브라우즈합니다.
2. **2단계: 대화 원문 상세 인출 (`session_recall` 도구 - Anchor & Lineage)**
   - 1단계에서 찾은 `session_id`와 관련 키워드를 `anchor_message`로 넘기면, 해당 세션의 전체 대화 중 가장 관련 있는 메시지(Anchor)를 찾아 전후 ±`window` 대화를 복원합니다.
   - **Bookends 보존**: 세션의 시작 맥락과 결론을 놓치지 않기 위해 **첫 3개 메시지 + 마지막 3개 메시지**를 항상 함께 반환합니다.

아래 실습에서는 10개 대형 시나리오 대화를 FTS5로 인덱싱하고, FTS5 검색 정확도 검증 및 2단계 도구 인출을 시연합니다.


In [9]:
import os
import json
import asyncio
from app.middleware.memory import EpisodicStore

# 1. artifacts/chat 폴더에서 10개 시나리오 대화 JSON 파일 자동 로드
chat_dir = "./artifacts/chat"
if not os.path.exists(chat_dir):
    chat_dir = "../artifacts/chat"

scenario_files = sorted([f for f in os.listdir(chat_dir) if f.endswith(".json")])
scenarios = []
for file_name in scenario_files:
    file_path = os.path.join(chat_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        scenarios.append(json.load(f))
print(f"📦 [artifacts/chat 디렉토리] 총 {len(scenarios)}개 대형 시나리오 대화 세션 로드 완료!\n")


📦 [artifacts/chat 디렉토리] 총 10개 대형 시나리오 대화 세션 로드 완료!



In [10]:
# 2. EpisodicStore 초기화, 파이널라이즈 및 2단계 도구 연계 인출 실습
from app.middleware.memory import MemoryMiddleware

async def test_episodic_with_artifacts():
    db_path = os.path.join(tmp_dir, "episodic_demo.db")
    episodic_store = EpisodicStore(db_path=db_path)
    await episodic_store.setup()
    print("=== [1. 10개 대형 세션 파이널라이즈 및 FTS5 인덱싱] ===")
    for sc in scenarios:
        sid = sc["session_id"]
        msgs = sc["messages"]
        summary = await episodic_store.finalize_session(sid, msgs, llm=llm)
        print(f"  - [{sid}] 요약 저장 완료: {summary[:75]}...")

    print("\n=== [2. FTS5 전문 검색을 통한 세션 탐색 (Session Discovery)] ===")
    queries = [
        ("marketing campaign server S3 CloudFront Slack", "scenario_01"),
        ("Oracle DB query customer_pii SHA-256 dashboard", "scenario_02"),
        ("K8s cluster HashiCorp Vault secrets retention", "scenario_03"),
        ("GPU fine-tuning PyTorch Weights & Biases DeepSpeed", "scenario_04"),
        ("payment gateway OAuth PCI-DSS audit log", "scenario_05"),
        ("E-Commerce Redis cart TTL Mutex Lock Kafka", "scenario_06"),
        ("Embedded IoT firmware MQTT ECDSA Dual Bank", "scenario_07"),
        ("Game server matchmaking MMR UDP KCP 60Hz", "scenario_08"),
        ("Healthcare DICOM PACS HIPAA AES-256 Lossless", "scenario_09"),
        ("Blockchain Ethereum Solidity The Graph Subgraph", "scenario_10"),
    ]
    matched_count = 0
    for q, expected_sid in queries:
        results = await episodic_store.search_sessions(query=q, top_k=1)
        if results:
            top_match = results[0]
            matched_sid = top_match["session_id"]
            if matched_sid == expected_sid:
                matched_count += 1
                print(f"🔎 쿼리: '{q[:40]}...' -> ✅ MATCHED! [{matched_sid}]")
            else:
                print(f"🔎 쿼리: '{q[:40]}...' -> ⚠️ MISMATCH [{matched_sid}] (Expected: {expected_sid})")
    print(f"\n📊 FTS5 세션 검색 매칭 정확도: {matched_count}/{len(queries)} ({int(matched_count/len(queries)*100)}%) 성공!")

    print("\n=== [3. Anchor 키워드 기반 세션 맥락 인출 (Lineage Retrieval)] ===")
    anchors = [
        ("scenario_01", "CloudFront"),
        ("scenario_06", "Mutex Lock"),
        ("scenario_09", "HIPAA"),
    ]
    for sid, anchor_kw in anchors:
        recalled = await episodic_store.get_anchored_view(sid, anchor_keyword=anchor_kw, window=1)
        print(f"\n📍 [{sid} Anchor '{anchor_kw}' 맥락 인출 결과]: 총 {len(recalled)}개 메시지 (Bookends + Core View)")
        for m in recalled[:3]:
            print(f"  - [{m['role']}] {m['content'][:65]}...")

    print("\n=== [4. 에이전트 도구 레벨 2단계 인출 실습 (session_search ➔ session_recall)] ===")
    demo_mw = MemoryMiddleware(semantic_store=semantic_store, episodic_store=episodic_store)
    
    # 1단계 도구 호출: session_search
    search_query = "Healthcare DICOM HIPAA compliance"
    print(f"🛠️ [1단계: session_search 호출] query='{search_query}'")
    search_json = demo_mw._session_search_tool.invoke({"query": search_query, "limit": 1})
    search_res = json.loads(search_json)
    target_sid = search_res["results"][0]["session_id"]
    print(f"   ↳ 탐색된 세션 ID: {target_sid} (요약: {search_res['results'][0]['summary'][:60]}...)")

    # 2단계 도구 호출: session_recall
    print(f"🛠️ [2단계: session_recall 호출] session_id='{target_sid}', anchor='HIPAA'")
    recall_json = demo_mw._session_recall_tool.invoke({"session_id": target_sid, "anchor_message": "HIPAA", "window": 1})
    recall_res = json.loads(recall_json)
    print(f"   ↳ 복원된 메시지 수: {recall_res['message_count']}개")
    for msg in recall_res["messages"][:2]:
        print(f"     • [{msg['role']}] {msg['content'][:60]}...")

    await episodic_store.close()
    
    if os.path.exists(db_path):
        os.remove(db_path)
        print("\n🧹 [episodic_demo.db Reset 초기화 완료]")

await test_episodic_with_artifacts()


=== [1. 10개 대형 세션 파이널라이즈 및 FTS5 인덱싱] ===
  - [scenario_01] 요약 저장 완료: The conversation involves the planning and setup of a new brand campaign in...
  - [scenario_02] 요약 저장 완료: The conversation revolves around data analysis and the use of Oracle DB que...
  - [scenario_03] 요약 저장 완료: The conversation revolves around establishing a deployment guide for a back...
  - [scenario_04] 요약 저장 완료: The conversation revolves around setting up an experimental environment for...
  - [scenario_05] 요약 저장 완료: The conversation involves discussions about integrating a new PG payment ga...
  - [scenario_06] 요약 저장 완료: The conversation involves the coordination of an e-commerce development tea...
  - [scenario_07] 요약 저장 완료: The conversation discusses the implementation of an OTA firmware update pip...
  - [scenario_08] 요약 저장 완료: The conversation revolves around the design of a real-time multiplayer game...
  - [scenario_09] 요약 저장 완료: The conversation revolves around the establishment of a healthcare solution

## 3. Memory Middleware & Closed Learning Loop

**MemoryMiddleware**는 에이전트 실행 수명주기(Lifecycle) 훅을 제어하고, 에이전트에 **3종 메모리 도구 번들**을 공급하는 핵심 컴포넌트입니다.

### Hermes 미들웨어 동작 원칙
1. **`before_agent` 훅 (Frozen Snapshot & Clean Slate)**:
   - **L3 Semantic Memory (`MEMORY.md` / `USER.md`)**: 세션 시작 시 디스크 스냅샷을 1회 읽어와 `AgentContext.recalled_memory`에 주입합니다. 세션 진행 중에는 스냅샷이 변경되지 않아 **GPU KV-Cache를 완벽히 보존**합니다.
   - **L2 Episodic Memory**: 매 턴 프롬프트에 자동 주입하지 않고 **Clean Slate**를 유지합니다. (불필요한 토큰 낭비 및 환각 방지)
2. **도구 팩토리 (`get_tools()`)**:
   - `memory()`: L3 장기 시맨틱 메모리 조작 도구 (add / replace / remove)
   - `session_search()`: L2 과거 세션 메타데이터 FTS5 검색 및 최근 목록 브라우즈 도구
   - `session_recall()`: L2 특정 세션의 대화 원문을 Anchor/Bookend 기반으로 정밀 인출하는 도구
3. **`after_agent` 훅 (Closed Learning Loop)**:
   - 에이전트의 답변 턴이 끝난 직후 **비동기 데몬 스레드(Daemon Thread)**를 생성합니다.
   - 별도의 리뷰 LLM이 최근 대화 스냅샷을 검토(Background Review)하여 신규 팩트를 추출한 뒤 `memory(action='add', ...)`를 자동 격발하고, 세션 요약을 `EpisodicStore`에 저장합니다.


In [15]:
from app.utils.context import AgentContext
from app.middleware.memory import MemoryMiddleware

# 메모리 옵션이 활성화된 AgentContext 정의
ctx = AgentContext(
    episodic_memory_enabled=True,
    semantic_memory_enabled=True,
    memory_learning_enabled=True,
    memory_dir=tmp_dir,
    episodic_db_path=os.path.join(tmp_dir, "episodic.db")
)

print("AgentContext Memory Configuration:")
print(f"- Episodic Memory Enabled: {ctx.episodic_memory_enabled}")
print(f"- Semantic Memory Enabled: {ctx.semantic_memory_enabled}")
print(f"- Closed Learning Loop Enabled: {ctx.memory_learning_enabled}")
print(f"- Recalled Memory Field: '{ctx.recalled_memory}'")

# MemoryMiddleware 생성 및 바인딩 도구 3종 확인
memory_mw = MemoryMiddleware(semantic_store=semantic_store, episodic_store=episodic_store, review_llm=llm)
tools = memory_mw.get_tools()

print("\n🛠️ [MemoryMiddleware 바인딩 3종 도구 세트]:")
for t in tools:
    first_doc_line = t.description.strip().split("\n")[0] if t.description else ""
    print(f"  • 도구명: {t.name:<16} | 설명: {first_doc_line}")


AgentContext Memory Configuration:
- Episodic Memory Enabled: True
- Semantic Memory Enabled: True
- Closed Learning Loop Enabled: True
- Recalled Memory Field: ''

🛠️ [MemoryMiddleware 바인딩 3종 도구 세트]:
  • 도구명: memory           | 설명: Manage the agent's long-term semantic memory (MEMORY.md / USER.md).
  • 도구명: session_search   | 설명: Search past conversation sessions to recall relevant context.
  • 도구명: session_recall   | 설명: Recall messages from a specific past session using anchor-based retrieval.


## 💡 요약 및 정리

1. **L1 Working Memory**: `AsyncSqliteSaver`를 통한 단기 세션 상태 보존.
2. **L2 Episodic Memory**: 세션 종료 후 요약 FTS5 인덱싱 및 2단계 도구 연계(`session_search` ➔ `session_recall`)를 통한 능동적 인출 (A안: Clean Slate 준수).
3. **L3 Semantic Memory**: `§` 구분자 기반의 마크다운 팩트 관리 및 Frozen Snapshot을 통한 GPU KV-Cache 보존과 상시 주입.
4. **Closed Learning Loop**: `after_agent` 데몬 스레드에서 메인 스레드 블로킹 없이 대화를 자동 리뷰하고 `memory()` 도구 자율 격발을 통해 지속적으로 진화하는 자율 메모리 루프 완성.
